# The core CubeDynamics grammar

This vignette is the smallest complete CubeDynamics workflow. It builds a
deterministic in-memory cube, composes public verbs, and unwraps the result.
It needs no network access, credentials, or local data.

## Build a small cube

CubeDynamics works with xarray objects. The conventional dimensions are
`time`, `y`, and `x`; the values can be NumPy- or Dask-backed.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

rng = np.random.default_rng(42)
time = pd.date_range("2024-01-01", periods=12, freq="MS")
season = np.sin(np.linspace(0, 2 * np.pi, time.size, endpoint=False))[:, None, None]
spatial = np.array([[0.0, 0.2, 0.4], [0.1, 0.3, 0.5]])[None, :, :]
noise = rng.normal(0, 0.02, size=(time.size, 2, 3))

cube = xr.DataArray(
    season + spatial + noise,
    dims=("time", "y", "x"),
    coords={"time": time, "y": [40.1, 40.0], "x": [-105.2, -105.1, -105.0]},
    name="environmental_signal",
    attrs={"units": "1", "source": "deterministic synthetic vignette"},
)
cube

## Compose verbs

The outer verb call stores configuration; the pipe passes the current value to
the returned callable. `unwrap()` marks the boundary where ordinary Python and
xarray use resumes.

In [ ]:
spatial_anomaly_series = (
    pipe(cube)
    | v.anomaly(dim="time")
    | v.mean(dim=("y", "x"), keep_dim=False)
).unwrap()

assert spatial_anomaly_series.dims == ("time",)
assert abs(float(spatial_anomaly_series.mean())) < 1e-12
spatial_anomaly_series

## The grammar is regular Python

A pipe may contain built-in verbs and ordinary callables. A reusable scientific
callable is usually moved into the project that owns its assumptions and given
direct-call and pipe tests.

In [ ]:
def name_result(name):
    def _op(value):
        return value.rename(name)
    return _op

named = (pipe(spatial_anomaly_series) | name_result("regional_anomaly")).unwrap()
assert named.name == "regional_anomaly"
named.to_dataframe().head()

The stable idea is the composition protocol—not this synthetic dataset. Replace
`cube` with a local xarray object, a maintained data adapter, or a project-owned
stream and keep the same grammar.